# scenedetect

In [28]:
from scenedetect import open_video, SceneManager, StatsManager
from scenedetect.detectors import ContentDetector

def rough_cut(video_path, threshold=25.0):
    video = open_video(video_path)
    stats_manager = StatsManager()
    scene_manager = SceneManager(stats_manager=stats_manager)
    scene_manager.add_detector(
        ContentDetector(threshold=threshold, min_scene_len=15)
    )
    scene_manager.detect_scenes(video)
    
    scene_list = scene_manager.get_scene_list()
    
    # 从 stats_manager 提取每帧的 content_val 分数
    # ContentDetector 存储的 metric key 是 'content_val'
    frame_scores = {}
    if stats_manager is not None:
        for frame_num, metrics in stats_manager._frame_metrics.items():
            score = metrics.get('content_val', 0)
            frame_scores[frame_num] = score
    
    segments = []
    for i, (start, end) in enumerate(scene_list):
        end_frame = end.frame_num
        # 取切割点附近几帧的最大分数作为该切割的置信度
        nearby_scores = [
            frame_scores.get(end_frame + offset, 0)
            for offset in range(-2, 3)
        ]
        cut_score = max(nearby_scores) if nearby_scores else 0
        
        segments.append({
            'index': i,
            'start_sec': start.seconds,
            'end_sec': end.seconds,
            'start_frame': start.frame_num,
            'end_frame': end_frame,
            'cut_score': cut_score,
            'duration': end.seconds - start.seconds
        })
    
    return segments

In [29]:
segments = rough_cut("videos/6.mp4")
segments

[{'index': 0,
  'start_sec': 0.0,
  'end_sec': 1.7,
  'start_frame': 0,
  'end_frame': 51,
  'cut_score': np.float64(87.4222728587963),
  'duration': 1.7},
 {'index': 1,
  'start_sec': 1.7,
  'end_sec': 2.5,
  'start_frame': 51,
  'end_frame': 75,
  'cut_score': np.float64(50.39539026331018),
  'duration': 0.8},
 {'index': 2,
  'start_sec': 2.5,
  'end_sec': 3.3,
  'start_frame': 75,
  'end_frame': 99,
  'cut_score': np.float64(26.15229853877315),
  'duration': 0.7999999999999998},
 {'index': 3,
  'start_sec': 3.3,
  'end_sec': 4.866667,
  'start_frame': 99,
  'end_frame': 146,
  'cut_score': np.float64(81.51916956018518),
  'duration': 1.5666669999999998},
 {'index': 4,
  'start_sec': 4.866667,
  'end_sec': 9.866667,
  'start_frame': 146,
  'end_frame': 296,
  'cut_score': np.float64(85.85337094907408),
  'duration': 5.0},
 {'index': 5,
  'start_sec': 9.866667,
  'end_sec': 10.4,
  'start_frame': 296,
  'end_frame': 312,
  'cut_score': np.float64(59.08396629050927),
  'duration': 0.53

In [30]:
import cv2
from scenedetect import split_video_ffmpeg
from scenedetect.frame_timecode import FrameTimecode

def split_video_by_segments(video_path, segments, output_dir="output_clips"):
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    
    scene_list = [
        (
            FrameTimecode(seg['start_sec'], fps),
            FrameTimecode(seg['end_sec'], fps)
        )
        for seg in segments
    ]
    
    split_video_ffmpeg(
        video_path,
        scene_list,
        output_dir=output_dir,
        output_file_template="clip_$SCENE_NUMBER.mp4"
    )
    
    print(f"共切割 {len(scene_list)} 个片段，保存至 {output_dir}/")

In [ ]:
segments

In [31]:
split_video_by_segments("videos/6.mp4", segments)

共切割 14 个片段，保存至 output_clips/


# AI

In [ ]:
import asyncio
import base64
import os
from pathlib import Path
from pydantic import BaseModel, Field
import instructor

# ── Pydantic 响应模型 ──────────────────────────────────────────────────────────

class CutPoint(BaseModel):
    timestamp: float = Field(description="切割时间点（秒）")
    reason: str = Field(description="切割原因")

class CutPointList(BaseModel):
    cut_points: list[CutPoint] = Field(description="所有切割时间点列表")

# ── Prompt ─────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """你是一个视频特效片段切割专家。
你的任务是分析视频，输出最合适的切割时间点列表，将视频切分为若干独立片段。

切割标准：
- 每个片段应完整包含针对单一对象（一段文字、一个图片、一个元素）的特效过程
- 切割点应在两个不同特效对象之间的间隙处，而非特效动画中途
- 特效动画过程中不应切割，即使画面变化剧烈
- 片段不应过短（低于 0.5 秒），也不应包含过多不同元素的特效（超过 2 个对象）
"""

USER_PROMPT = """请分析这段视频（总时长 {duration:.2f} 秒），输出切割时间点列表。

注意：
- 不需要输出视频的起始点（0秒）和结束点（{duration:.2f}秒）
- 只输出中间的切割时间点
- 时间点精确到小数点后两位
"""

# ── 获取视频时长 ───────────────────────────────────────────────────────────────

def get_video_duration(video_path: str) -> float:
    import cv2
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    return frame_count / fps

# ── LLM 直接判断切割点 ────────────────────────────────────────────────────────

async def detect_cut_points_async(
    video_path: str,
    async_client,
) -> CutPointList:
    instructor_client = instructor.from_openai(async_client)

    duration = get_video_duration(video_path)
    video_b64 = video_to_base64(video_path)

    response = await instructor_client.chat.completions.create(
        model=os.getenv("MODEL"),
        response_model=CutPointList,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": USER_PROMPT.format(duration=duration)},
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/mp4;base64,{video_b64}"},
                    },
                ],
            },
        ],
    )
    return response

# ── 将切割点转为 segments ──────────────────────────────────────────────────────

def cut_points_to_segments(cut_points: CutPointList, duration: float) -> list[dict]:
    timestamps = sorted([cp.timestamp for cp in cut_points.cut_points])
    
    # 加入首尾
    boundaries = [0.0] + timestamps + [duration]
    
    segments = []
    for i in range(len(boundaries) - 1):
        start = boundaries[i]
        end = boundaries[i + 1]
        segments.append({
            "index": i,
            "start_sec": round(start, 2),
            "end_sec": round(end, 2),
            "duration": round(end - start, 2),
        })
    
    return segments

# ── 用 ffmpeg 按 segments 切割视频 ────────────────────────────────────────────

def split_by_segments(video_path: str, segments: list[dict], output_dir: str = "output_clips"):
    import subprocess
    os.makedirs(output_dir, exist_ok=True)

    for seg in segments:
        output_path = os.path.join(output_dir, f"clip_{seg['index']:03d}.mp4")
        cmd = [
            "ffmpeg", "-y",
            "-ss", str(seg["start_sec"]),
            "-i", video_path,
            "-t", str(seg["duration"]),
            "-c", "copy",
            output_path,
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        print(f"  clip_{seg['index']:03d}.mp4  {seg['start_sec']}s - {seg['end_sec']}s  ({seg['duration']}s)")

# ── 完整管线 ──────────────────────────────────────────────────────────────────

async def run_pipeline_async(
    video_path: str,
    async_client,
    output_dir: str = "output_clips",
):
    print("Step 1: LLM 分析视频切割点...")
    cut_point_list = await detect_cut_points_async(video_path, async_client)

    print(f"  LLM 输出 {len(cut_point_list.cut_points)} 个切割点：")
    for cp in cut_point_list.cut_points:
        print(f"    {cp.timestamp:.2f}s — {cp.reason}")

    duration = get_video_duration(video_path)
    segments = cut_points_to_segments(cut_point_list, duration)
    print(f"\nStep 2: 生成 {len(segments)} 个片段")

    print("\nStep 3: 切割视频...")
    split_by_segments(video_path, segments, output_dir=output_dir)

    print(f"\n完成，片段保存至 {output_dir}/")
    return segments

In [ ]:
from openai import AsyncOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

async_client = AsyncOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
)

segments = await run_pipeline_async("videos/6.mp4", async_client)